# TPUGraphs — Final Submission Notebook

Produces `submission.csv` (all 5 collections) for
**Google – Fast or Slow? Predict AI Model Runtime**, from the trained GNN checkpoints.

## How to run on Kaggle
1. **Create → Notebook**, then *File → Import Notebook* and upload this file
   (or *Copy & Edit* from the repo).
2. **Add Data** (right panel):
   - the **competition** `predict-ai-model-runtime` (the test graphs), and
   - the **checkpoints**: either add the output of the training kernel
     `tpugraphs-layout-training-phase-3` (*Add Data → Your Work → Notebook Output*),
     **or** a dataset containing the five `*_sage/best.pt` files. The tile checkpoint
     (`tile_sage_listmle` or `tile_sage_hinge`) must be included too.
3. **Settings:** Internet **ON** (to clone the code repo). GPU optional — inference is
   light enough for CPU, GPU just makes it faster.
4. **Run All.** The last cell writes `/kaggle/working/submission.csv`.
5. **Submit:** *Save Version → Submit to Competition* (uses the notebook's
   `submission.csv` output), or download it and use *Submit Predictions* on the
   competition page.

The notebook auto-discovers checkpoints anywhere under `/kaggle/input`, so the exact
dataset name does not matter as long as each `<run>_sage/best.pt` is attached.

In [ ]:
# 1. Environment: pin the torch/PyG pair the repo needs (Kaggle's default
#    torch 2.10 lacks P100 sm_60 kernels AND breaks torch-geometric's dynamo guards).
import os, subprocess, sys
def sh(cmd):
    print("$", cmd, flush=True); subprocess.run(cmd, shell=True, check=True)
sh(f"{sys.executable} -m pip -q uninstall -y torch torchvision torchaudio")
sh(f"{sys.executable} -m pip -q install torch==2.4.1 --index-url https://download.pytorch.org/whl/cu118")
sh(f"{sys.executable} -m pip -q install --no-deps torch-geometric==2.6.1")

REPO = "https://github.com/Bobksl/Predict-AI-Model-Runtime.git"
REPO_DIR = "/kaggle/working/prj"
if not os.path.exists(REPO_DIR):
    sh(f"git clone --depth 1 {REPO} {REPO_DIR}")
os.chdir(REPO_DIR); sys.path.insert(0, REPO_DIR)
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())

In [ ]:
# 2. Locate the 5 checkpoints (search everything mounted under /kaggle/input).
import glob
from pathlib import Path

COLLECTIONS = {
    "tile:xla":            ["tile_sage_listmle", "tile_sage_hinge", "tile_sage"],
    "layout:xla:random":   ["layout_xla_random_sage"],
    "layout:xla:default":  ["layout_xla_default_sage"],
    "layout:nlp:random":   ["layout_nlp_random_sage"],
    "layout:nlp:default":  ["layout_nlp_default_sage"],
}
SEARCH_ROOTS = ["/kaggle/input", "/kaggle/working",
                str(Path(REPO_DIR) / "artifacts" / "checkpoints"), "artifacts/checkpoints"]

def find_ckpt(run_names):
    for run in run_names:
        for base in SEARCH_ROOTS:
            hits = glob.glob(f"{base}/**/{run}/best.pt", recursive=True)
            if hits:
                return sorted(hits)[0]
    return None

ckpts = {coll: find_ckpt(runs) for coll, runs in COLLECTIONS.items()}
for coll, p in ckpts.items():
    print(f"{coll:22s} -> {p}")
missing = [c for c, p in ckpts.items() if p is None]
assert not missing, (f"Missing checkpoints for {missing}. Attach a dataset / notebook "
                     f"output containing the corresponding <run>_sage/best.pt files.")

In [ ]:
# 3. Resolve the competition data root (local checkout or Kaggle mount).
from src.data.paths import resolve_data_root, list_npz
DATA = resolve_data_root()
print("data root:", DATA)

In [ ]:
# 4. Inference: score every configuration of every test graph, per collection.
import torch
from src.data.normalize import NodeFeatNormalizer
from src.models import build_model
from src.inference.predict import score_all_configs, rank_configs
from src.inference.predict_layout import score_all_configs_layout

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
norm = NodeFeatNormalizer.load(str(Path(REPO_DIR) / "artifacts" / "norm_stats.json"))

rows = {}   # "collection:stem" -> ranked config indices (fastest first)
for coll, ckpt_path in ckpts.items():
    ck = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    cfg = ck["config"]
    model = build_model(cfg["model"]).to(device)
    model.load_state_dict(ck["state_dict"]); model.eval()
    rev = cfg["data"].get("add_reverse_edges", True)
    files = list_npz(DATA, coll, "test")
    print(f"{coll:22s} {len(files):3d} test files | ckpt val_opa={ck['best_val_opa']:.4f}", flush=True)
    for fp in files:
        if coll.startswith("tile"):
            s = score_all_configs(model, str(fp), norm, device, add_reverse_edges=rev)
        else:
            s = score_all_configs_layout(model, str(fp), norm, device, add_reverse_edges=rev)
        rows[f"{coll}:{fp.stem}"] = rank_configs(s)
print("total ranked graphs:", len(rows))

In [ ]:
# 5. Assemble + validate + write submission.csv.
import csv
from src.data.cache import read_bundle

# validate: every ranking is a permutation of that graph's configs
bad = 0
for coll, ckpt_path in ckpts.items():
    for fp in list_npz(DATA, coll, "test"):
        key = f"{coll}:{fp.stem}"
        n = int(read_bundle(str(fp))["config_runtime"].shape[0])
        if sorted(rows[key]) != list(range(n)):
            bad += 1
assert bad == 0, f"{bad} rows are not valid permutations"

OUT = "/kaggle/working/submission.csv"
with open(OUT, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["ID", "TopConfigs"])
    for key in sorted(rows):
        w.writerow([key, ";".join(map(str, rows[key]))])
print(f"WROTE {OUT} with {len(rows)} rows — all validated as permutations.")
print("Preview:")
print(open(OUT).readline().strip())
print(open(OUT).readlines()[1][:90], "...")